In [27]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import requests
import json
import time
import getpass

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/a2"

models_path = os.path.join(BASE_DIR, "models_config.xlsx")
prompt1_path = os.path.join(BASE_DIR, "prompt1.txt")
raw_prompt1_save_path = os.path.join(BASE_DIR, "raw_prompt1_outputs.xlsx")

models_df = pd.read_excel(models_path)

with open(prompt1_path, "r", encoding="utf-8") as f:
    PROMPT_1 = f.read()

print("BASE_DIR:", BASE_DIR)
print("models_df shape:", models_df.shape)
print(models_df.head())
print("Prompt 1 length:", len(PROMPT_1))

BASE_DIR: /content/drive/MyDrive/Colab Notebooks/a2
models_df shape: (15, 5)
     provider                      base_url         api_key_env  \
0  OpenRouter  https://openrouter.ai/api/v1  OPENROUTER_API_KEY   
1  OpenRouter  https://openrouter.ai/api/v1  OPENROUTER_API_KEY   
2  OpenRouter  https://openrouter.ai/api/v1  OPENROUTER_API_KEY   
3    Together   https://api.together.xyz/v1    TOGETHER_API_KEY   
4    Together   https://api.together.xyz/v1    TOGETHER_API_KEY   

                                      model enabled  
0  mistralai/mistral-small-3.1-24b-instruct     yes  
1                     google/gemma-3-12b-it     yes  
2          meta-llama/llama-3.1-8b-instruct     yes  
3            Qwen/Qwen2.5-7B-Instruct-Turbo     yes  
4  meta-llama/Meta-Llama-3-8B-Instruct-Lite     yes  
Prompt 1 length: 1086


In [29]:
OPENROUTER_API_KEY = getpass.getpass("Enter OPENROUTER_API_KEY (or press Enter to skip): ")
TOGETHER_API_KEY = getpass.getpass("Enter TOGETHER_API_KEY (or press Enter to skip): ")
GROQ_API_KEY = getpass.getpass("Enter GROQ_API_KEY (or press Enter to skip): ")
SAMBANOVA_API_KEY = getpass.getpass("Enter SAMBANOVA_API_KEY (or press Enter to skip): ")
DEEPINFRA_API_KEY = getpass.getpass("Enter DEEPINFRA_API_KEY (or press Enter to skip): ")

api_keys = {
    "OPENROUTER_API_KEY": OPENROUTER_API_KEY,
    "TOGETHER_API_KEY": TOGETHER_API_KEY,
    "GROQ_API_KEY": GROQ_API_KEY,
    "SAMBANOVA_API_KEY": SAMBANOVA_API_KEY,
    "DEEPINFRA_API_KEY": DEEPINFRA_API_KEY
}

print({k: ("loaded" if v else "empty") for k, v in api_keys.items()})

Enter OPENROUTER_API_KEY (or press Enter to skip): ··········
Enter TOGETHER_API_KEY (or press Enter to skip): ··········
Enter GROQ_API_KEY (or press Enter to skip): ··········
Enter SAMBANOVA_API_KEY (or press Enter to skip): ··········
Enter DEEPINFRA_API_KEY (or press Enter to skip): ··········
{'OPENROUTER_API_KEY': 'empty', 'TOGETHER_API_KEY': 'empty', 'GROQ_API_KEY': 'empty', 'SAMBANOVA_API_KEY': 'loaded', 'DEEPINFRA_API_KEY': 'empty'}


In [30]:
PROMPT1_RUNS_PER_MODEL = 2
TEMPERATURE = 0.7
MAX_TOKENS = 1200
WAIT_SECONDS_BETWEEN_CALLS = 3

enabled_models = models_df[models_df["enabled"].str.lower() == "yes"].copy()

print("Enabled models:", len(enabled_models))
enabled_models[["provider", "model", "api_key_env"]]

Enabled models: 15


,provider,model,api_key_env
0,OpenRouter,mistralai/mistral-small-3.1-24b-instruct,OPENROUTER_API_KEY
1,OpenRouter,google/gemma-3-12b-it,OPENROUTER_API_KEY
2,OpenRouter,meta-llama/llama-3.1-8b-instruct,OPENROUTER_API_KEY
3,Together,Qwen/Qwen2.5-7B-Instruct-Turbo,TOGETHER_API_KEY
4,Together,meta-llama/Meta-Llama-3-8B-Instruct-Lite,TOGETHER_API_KEY
5,Together,openai/gpt-oss-20b,TOGETHER_API_KEY
6,Groq,llama-3.1-8b-instant,GROQ_API_KEY
7,Groq,llama-3.3-70b-versatile,GROQ_API_KEY
8,Groq,openai/gpt-oss-20b,GROQ_API_KEY
9,SambaNova,Meta-Llama-3.3-70B-Instruct,SAMBANOVA_API_KEY


In [31]:
def call_chat_completion(base_url, api_key, model, prompt, temperature=0.7, max_tokens=1200, retries=3, wait_seconds=10):
    url = f"{base_url}/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    data = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    for attempt in range(retries):
        response = requests.post(url, headers=headers, data=json.dumps(data))
        print("HTTP status:", response.status_code)

        try:
            result = response.json()
        except Exception:
            print("Raw response:", response.text)
            return None, response.status_code

        if response.status_code == 429:
            print(f"Rate limited. Retry {attempt + 1}/{retries} after {wait_seconds} seconds...")
            time.sleep(wait_seconds)
            continue

        if response.status_code >= 400:
            err_msg = result.get("error", {}).get("message", str(result))
            print("Error:", err_msg)
            return None, response.status_code

        return result["choices"][0]["message"]["content"], response.status_code

    print("Failed after retries.")
    return None, 429

In [32]:
if os.path.exists(raw_prompt1_save_path):
    raw_prompt1_df = pd.read_excel(raw_prompt1_save_path)
    print("Loaded existing raw_prompt1_outputs.xlsx")
else:
    raw_prompt1_df = pd.DataFrame(columns=[
        "provider", "base_url", "model", "prompt1_run_id", "raw_text", "status"
    ])
    print("No existing file found. Starting fresh.")

print("Current saved rows:", raw_prompt1_df.shape[0])
raw_prompt1_df.tail()

Loaded existing raw_prompt1_outputs.xlsx
Current saved rows: 30


,provider,base_url,model,prompt1_run_id,raw_text,status
25,DeepInfra,https://api.deepinfra.com/v1/openai,Qwen/Qwen3-30B-A3B,2,"<think>\nOkay, let's tackle this query. The us...",200
26,DeepInfra,https://api.deepinfra.com/v1/openai,mistralai/Mistral-Small-3.2-24B-Instruct-2506,1,Here are three diverse personas for your virtu...,200
27,DeepInfra,https://api.deepinfra.com/v1/openai,mistralai/Mistral-Small-3.2-24B-Instruct-2506,2,Here are three diverse personas for your virtu...,200
28,DeepInfra,https://api.deepinfra.com/v1/openai,Qwen/Qwen3-Coder-30B-A3B-Instruct,1,"<think>\nOkay, the user wants three personas f...",200
29,DeepInfra,https://api.deepinfra.com/v1/openai,Qwen/Qwen3-Coder-30B-A3B-Instruct,2,"<think>\nOkay, let's tackle this query. The us...",200


In [33]:
for _, row in enabled_models.iterrows():
    provider = row["provider"]
    base_url = row["base_url"]
    api_key_env = row["api_key_env"]
    model_name = row["model"]

    api_key = api_keys.get(api_key_env, "")

    if not api_key:
        print(f"Skipping {model_name} because {api_key_env} is empty.")
        print("-" * 80)
        continue

    for run_id in range(1, PROMPT1_RUNS_PER_MODEL + 1):
        already_done = (
            (raw_prompt1_df["provider"] == provider) &
            (raw_prompt1_df["model"] == model_name) &
            (raw_prompt1_df["prompt1_run_id"] == run_id)
        ).any()

        if already_done:
            print(f"Skipping existing result | {provider} | {model_name} | run {run_id}")
            continue

        print(f"Running Prompt 1 | Provider: {provider} | Model: {model_name} | Run: {run_id}")

        raw_text, status_code = call_chat_completion(
            base_url=base_url,
            api_key=api_key,
            model=model_name,
            prompt=PROMPT_1,
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS
        )

        new_row = pd.DataFrame([{
            "provider": provider,
            "base_url": base_url,
            "model": model_name,
            "prompt1_run_id": run_id,
            "raw_text": raw_text,
            "status": status_code
        }])

        raw_prompt1_df = pd.concat([raw_prompt1_df, new_row], ignore_index=True)

        raw_prompt1_df.to_excel(raw_prompt1_save_path, index=False)

        if raw_text:
            print(raw_text[:1000])
        else:
            print("No output returned.")

        print(f"Saved progress to: {raw_prompt1_save_path}")
        print("-" * 80)

        time.sleep(WAIT_SECONDS_BETWEEN_CALLS)

Skipping mistralai/mistral-small-3.1-24b-instruct because OPENROUTER_API_KEY is empty.
--------------------------------------------------------------------------------
Skipping google/gemma-3-12b-it because OPENROUTER_API_KEY is empty.
--------------------------------------------------------------------------------
Skipping meta-llama/llama-3.1-8b-instruct because OPENROUTER_API_KEY is empty.
--------------------------------------------------------------------------------
Skipping Qwen/Qwen2.5-7B-Instruct-Turbo because TOGETHER_API_KEY is empty.
--------------------------------------------------------------------------------
Skipping meta-llama/Meta-Llama-3-8B-Instruct-Lite because TOGETHER_API_KEY is empty.
--------------------------------------------------------------------------------
Skipping openai/gpt-oss-20b because TOGETHER_API_KEY is empty.
--------------------------------------------------------------------------------
Skipping llama-3.1-8b-instant because GROQ_API_KEY is emp

In [34]:
raw_prompt1_df = pd.read_excel(raw_prompt1_save_path)

print("Final shape:", raw_prompt1_df.shape)
print(raw_prompt1_df["status"].value_counts(dropna=False))
raw_prompt1_df[["provider", "model", "prompt1_run_id", "status"]].head(30)

Final shape: (32, 6)
status
200    30
410     2
Name: count, dtype: int64


,provider,model,prompt1_run_id,status
0,OpenRouter,mistralai/mistral-small-3.1-24b-instruct,1,200
1,OpenRouter,mistralai/mistral-small-3.1-24b-instruct,2,200
2,OpenRouter,google/gemma-3-12b-it,1,200
3,OpenRouter,google/gemma-3-12b-it,2,200
4,OpenRouter,meta-llama/llama-3.1-8b-instruct,1,200
5,OpenRouter,meta-llama/llama-3.1-8b-instruct,2,200
6,Together,Qwen/Qwen2.5-7B-Instruct-Turbo,1,200
7,Together,Qwen/Qwen2.5-7B-Instruct-Turbo,2,200
8,Together,meta-llama/Meta-Llama-3-8B-Instruct-Lite,1,200
9,Together,meta-llama/Meta-Llama-3-8B-Instruct-Lite,2,200


In [35]:
clean_prompt1_df = raw_prompt1_df[
    (raw_prompt1_df["status"] == 200) &
    (raw_prompt1_df["raw_text"].notna())
].copy()

clean_prompt1_df = clean_prompt1_df.drop_duplicates(
    subset=["provider", "model", "prompt1_run_id"],
    keep="last"
)

clean_save_path = os.path.join(BASE_DIR, "raw_prompt1_outputs_clean.xlsx")
clean_prompt1_df.to_excel(clean_save_path, index=False)

print("Saved clean file:", clean_save_path)
print("Clean shape:", clean_prompt1_df.shape)
clean_prompt1_df[["provider", "model", "prompt1_run_id", "status"]].head(30)

Saved clean file: /content/drive/MyDrive/Colab Notebooks/a2/raw_prompt1_outputs_clean.xlsx
Clean shape: (30, 6)


,provider,model,prompt1_run_id,status
0,OpenRouter,mistralai/mistral-small-3.1-24b-instruct,1,200
1,OpenRouter,mistralai/mistral-small-3.1-24b-instruct,2,200
2,OpenRouter,google/gemma-3-12b-it,1,200
3,OpenRouter,google/gemma-3-12b-it,2,200
4,OpenRouter,meta-llama/llama-3.1-8b-instruct,1,200
5,OpenRouter,meta-llama/llama-3.1-8b-instruct,2,200
6,Together,Qwen/Qwen2.5-7B-Instruct-Turbo,1,200
7,Together,Qwen/Qwen2.5-7B-Instruct-Turbo,2,200
8,Together,meta-llama/Meta-Llama-3-8B-Instruct-Lite,1,200
9,Together,meta-llama/Meta-Llama-3-8B-Instruct-Lite,2,200


In [38]:
error_summary = raw_prompt1_df.groupby(
    ["provider", "model", "status"]
).size().reset_index(name="count")

print(error_summary.sort_values(["provider", "model", "status"]))

      provider                                          model  status  count
0    DeepInfra                             Qwen/Qwen3-30B-A3B     200      2
1    DeepInfra              Qwen/Qwen3-Coder-30B-A3B-Instruct     200      2
2    DeepInfra  mistralai/Mistral-Small-3.2-24B-Instruct-2506     200      2
3         Groq                           llama-3.1-8b-instant     200      2
4         Groq                        llama-3.3-70b-versatile     200      2
5         Groq                             openai/gpt-oss-20b     200      2
6   OpenRouter                          google/gemma-3-12b-it     200      2
7   OpenRouter               meta-llama/llama-3.1-8b-instruct     200      2
8   OpenRouter       mistralai/mistral-small-3.1-24b-instruct     200      2
9    SambaNova                  DeepSeek-R1-Distill-Llama-70B     410      2
10   SambaNova                     Meta-Llama-3.1-8B-Instruct     200      2
11   SambaNova                    Meta-Llama-3.3-70B-Instruct     200      2